# Phase 1 — Generation with citations

Last stage of the vertical slice: retrieved chunks → **one** generation call → an answer whose
every claim resolves to a source. Input is the Qdrant collection from
`embed_index_phase1.ipynb`.

**This notebook imports `rag/` rather than defining anything.** The CLI runs the same objects,
so what gets eyeballed here is what ships — a helper copy-pasted into a notebook drifts from the
one in the CLI within a day, and then the write-up describes neither.

| Decision | Why |
|---|---|
| One call, no agent | ROADMAP ablation row 1. Rows 4–7 add planning, fanout, web search; each has to beat this, so the floor must be honestly a floor |
| Model cites `[n]`, never an arXiv ID | A model reproducing `2607.28503v1` from context will eventually emit `2607.28530v1` — a citation that looks perfect and resolves to nothing. Handles are minted by code and resolved by code |
| `INSUFFICIENT_CONTEXT` sentinel | Phase 10④ measures abstention on ~40 unanswerable questions. Detecting that by phrase-matching prose is a losing game; ask for a token instead |
| `temperature=0` | Ablation arms are compared by their deltas. A sampling generator makes part of every delta noise — you would be measuring the temperature |
| Uncited sources recorded every run | The observable signal for *retrieved-and-ignored* (ROADMAP §3①). Free now, unreconstructable later |
| Figure bytes attached at answer time | ROADMAP §1's multimodal claim has two halves. Retrieval matches the VLM caption; the generator gets the actual image |

In [1]:
import sys
from pathlib import Path

# The package lives at the repo root, not inside notebooks/.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from rag import config
from rag.generation import Generator, build_message, format_context, parse_citations
from rag.retrieval import Retriever

print("collection :", config.COLLECTION, f"({config.QDRANT_MODE})")
print("embedder   :", config.EMBED_MODEL_ID)
print("generator  :", config.GEN_MODEL_ID, f"temperature={config.GEN_TEMPERATURE}")
print("images     :", config.IMAGES_DIR, config.IMAGES_DIR.exists())

collection : arxiv_phase1 (local)
embedder   : BAAI/bge-m3
generator  : gemini-3.1-flash-lite temperature=0.0
images     : D:\.tutorials\agentic-rag-capstone\data\phase1\images True


## Retrieve

`Retriever` holds the embedder and the Qdrant client. Loading BGE-M3 is the slow part, so build
it once and reuse it for the whole session.

Embedded Qdrant takes an **exclusive lock** on its storage directory — while this notebook holds
it, `python -m rag.cli` will refuse to start. `retriever.close()` at the bottom releases it.

In [2]:
retriever = Retriever()

QUESTION = "How do multi-agent systems adapt their communication topology?"
chunks = retriever.search(QUESTION, top_k=5)

for handle, c in enumerate(chunks, 1):
    print(f"[{handle}] {c.score:.4f}  {c.provenance()}")
    print(f"     {' '.join(c.text.split())[:160]}...")

d:\.tutorials\agentic-rag-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 23238.95it/s]


[1] 0.7351  2607.28527v1 <ai> (text: abstract_intro)
     Abstract Large language model-based multi-agent systems improve complex problem solving through task decomposition, agent specialization, information exchange, ...
[2] 0.6996  2607.28527v1 <ai> (text: abstract_intro)
     ual agents rather than how multiple agents are organized and communicate. Multi-agent frameworks decompose complex tasks across specialized roles and collaborat...
[3] 0.6915  2607.28527v1 <ai> (text: abstract_intro)
     self-improve online therefore remains underexplored. This raises a central question: Can a multi-agent system improve its communication topology while solving e...
[4] 0.6876  2607.28527v1 <ai> (text: conclusion)
     Conclusion We presented MANTA, a framework that enables multi-agent systems to revise their collaboration topology during task execution. By combining topology ...
[5] 0.6868  2607.28527v1 <ai> (text: abstract_intro)
     L1 Prompt APO, APE, DSPy, MIPRO, PromptBreeder, TextGrad, 

## The prompt the model actually sees

Worth reading once rather than trusting. Two things to check: every source carries provenance in
its header, and no arXiv ID is presented in a position that invites the model to cite it as a
string.

In [3]:
print(format_context(chunks)[:2000])

[1] 2607.28527v1 <ai> - MANTA: Multi-Agent Network Topology Adaptation for Self-Evolving Multi-Agent Systems
    section: abstract_intro | type: text
Abstract
Large language model-based multi-agent systems improve
complex problem solving through task decomposition, agent
specialization, information exchange, and intermediate valida-
tion. However, existing systems typically treat communication
topology as a fixed design choice or an offline optimization
target. We introduce MANTA, a framework for Multi-Agent
Network Topology Adaptation that enables communication
structures to self-evolve at inference time. Before execution,
MANTA initializes a task-conditioned topology from prior
structural experience. During deployment, it monitors collab-
oration traces and applies bounded structural updates when
the current organization becomes insufficient. These updates
can modify agent roles, communication links, execution order,
information visibility, and validation pathways while preserv-
ing 

## Generate

One call. `Answer` carries the text plus everything needed to audit it — which handles were
cited, which were provided and ignored, which don't exist, whether the model abstained, how many
images went along, and the token counts.

In [4]:
generator = Generator()
answer = generator.generate(QUESTION, chunks)

print(answer.text)
print("\n" + "-" * 70)
print("cited    :", answer.cited)
print("uncited  :", answer.uncited, " <- retrieved, handed over, never referenced")
print("invalid  :", answer.invalid, " <- handles that do not exist (should be empty)")
print("abstained:", answer.abstained)
print("images   :", answer.images_attached, "attached")
print("usage    :", answer.usage, f"| {answer.latency_s:.1f}s")

Multi-agent systems typically treat communication topology as a fixed design choice or an offline optimization target, where structures are determined before task execution [1][2]. In contrast, the MANTA framework enables systems to self-evolve their topology during inference time [1]. This process involves planning an initial topology based on prior experience, monitoring collaboration traces during deployment, and applying bounded structural mutations when the current organization becomes insufficient [1][3][4]. These adaptations can include rewiring communication links, modifying agent roles, changing execution order, or adding validation pathways [1][4].

----------------------------------------------------------------------
cited    : [1, 2, 3, 4]
uncited  : [5]  <- retrieved, handed over, never referenced
invalid  : []  <- handles that do not exist (should be empty)
abstained: False
images   : 0 attached
usage    : {'input_tokens': 1468, 'output_tokens': 115, 'total_tokens': 1583

In [5]:
# Resolve every citation back to its source. This is the step that makes a citation a citation
# rather than a decoration - if it cannot round-trip here, it is not evidence.
for handle, chunk in answer.sources():
    print(f"[{handle}] {chunk.provenance()}")
    print(f"     {chunk.topic[:100]}")
    print(f"     score={chunk.score:.4f}  section_source={chunk.section_source}")

assert not answer.invalid, f"answer cites non-existent sources: {answer.invalid}"
print("\nall citations resolve")

[1] 2607.28527v1 <ai> (text: abstract_intro)
     MANTA: Multi-Agent Network Topology Adaptation for Self-Evolving Multi-Agent Systems
     score=0.7351  section_source=headings
[2] 2607.28527v1 <ai> (text: abstract_intro)
     MANTA: Multi-Agent Network Topology Adaptation for Self-Evolving Multi-Agent Systems
     score=0.6996  section_source=headings
[3] 2607.28527v1 <ai> (text: abstract_intro)
     MANTA: Multi-Agent Network Topology Adaptation for Self-Evolving Multi-Agent Systems
     score=0.6915  section_source=headings
[4] 2607.28527v1 <ai> (text: conclusion)
     MANTA: Multi-Agent Network Topology Adaptation for Self-Evolving Multi-Agent Systems
     score=0.6876  section_source=headings

all citations resolve


## The multimodal path

Retrieval matches a figure's **VLM caption**; generation gets the **original image**. Only the
first half existed before this notebook — ROADMAP §1 claims both, so here is the second half
being exercised rather than asserted.

`modality="figure"` forces figure chunks into the context so the path is actually taken. Note
what that costs: 16 of 544 chunks are figures, so an unfiltered question rarely retrieves one.

In [6]:
FIGURE_Q = "What does the figure comparing abstraction approaches show?"
fig_chunks = retriever.search(FIGURE_Q, top_k=3, modality="figure")

for handle, c in enumerate(fig_chunks, 1):
    path = c.image_path()
    print(f"[{handle}] {c.score:.4f} {c.provenance()}")
    print(f"     image: {path.name if path else 'MISSING'} "
          f"({path.stat().st_size if path else 0} bytes)")

# build_message reports any figure whose bytes cannot be found rather than quietly sending text
# only. A silently text-only 'multimodal' pipeline is the same failure shape as the captioning
# bug in PHASE1_NOTES 3.1 - it succeeds, and it drops a modality.
_, attached = build_message(FIGURE_Q, fig_chunks)
print(f"\n{attached}/{len(fig_chunks)} figure images attached to the request")

[1] 0.6127 2607.28498v1 <ai> (figure: abstract_intro, p2) 2607.28498v1_p1_i0
     image: 2607.28498v1_p1_i0.png (192251 bytes)
[2] 0.5280 2607.28526v1 <ai> (figure: conclusion, p8) 2607.28526v1_p7_i11
     image: 2607.28526v1_p7_i11.jpeg (13987 bytes)
[3] 0.5269 2607.28623v1 <ai> (figure: abstract_intro, p1) 2607.28623v1_p0_i0
     image: 2607.28623v1_p0_i0.png (3728733 bytes)

3/3 figure images attached to the request


In [7]:
fig_answer = generator.generate(FIGURE_Q, fig_chunks)
print(fig_answer.text)
print(f"\ncited={fig_answer.cited} images={fig_answer.images_attached} "
      f"usage={fig_answer.usage}")

The figure compares three approaches to analogical reasoning: direct matching, target-agnostic abstraction, and target-conditioned abstraction [1]. It illustrates that direct matching lacks an explicit basis for matches, while target-agnostic abstraction results in ambiguous relevance to the target [1]. In contrast, the proposed target-conditioned abstraction method uses an LLM to generate an explicit, interpretable, and target-specific transferable principle [1].

cited=[1] images=3 usage={'input_tokens': 3883, 'output_tokens': 86, 'total_tokens': 3969, 'input_token_details': {'cache_read': 0}}


In [8]:
# Same question, images withheld. If the answers are indistinguishable, the images are not
# earning their tokens and "multimodal" is a caption-retrieval system with extra cost - which is
# a finding worth having early, not a thing to discover in Phase 10.
text_only = generator.generate(FIGURE_Q, fig_chunks, include_images=False)
print(text_only.text)
print(f"\nwith images : {fig_answer.usage.get('input_tokens', '?')} input tokens")
print(f"without     : {text_only.usage.get('input_tokens', '?')} input tokens")

The figure comparing abstraction approaches illustrates three distinct methods: direct matching, target-agnostic abstraction, and the proposed target-conditioned abstraction [1]. It demonstrates that while the first two methods either fail to provide explicit reasoning or result in ambiguous abstractions, the proposed method utilizes a target-conditioned LLM [1]. This approach generates an explicit and interpretable transferable principle between a target problem and a candidate inspiration [1].

with images : 3883 input tokens
without     : 590 input tokens


## Abstention

The failure that matters most. A system at 0.92 faithfulness on answerable questions that
confidently fabricates on unanswerable ones is not production grade (ROADMAP §3④), and the only
way to know which one you have is to ask something the corpus cannot answer.

The corpus here is 20 `cs.AI` papers. A question about protein folding or 18th-century naval
history has no support in it, and the correct behaviour is to say so.

In [9]:
UNANSWERABLE = "What was the average grain yield per hectare in Prussia in 1840?"
oos_chunks = retriever.search(UNANSWERABLE, top_k=5)

print("retrieval still returns its top-5 - similarity is relative, not absolute:")
for handle, c in enumerate(oos_chunks, 1):
    print(f"  [{handle}] {c.score:.4f} {c.arxiv_id} {c.topic[:60]}")

oos = generator.generate(UNANSWERABLE, oos_chunks)
print(f"\nabstained: {oos.abstained}")
print(oos.text)

retrieval still returns its top-5 - similarity is relative, not absolute:
  [1] 0.3423 2607.28628v1 Learning to Trace Seiberg Dualities
  [2] 0.3395 2607.28628v1 Learning to Trace Seiberg Dualities
  [3] 0.3345 2607.28609v1 OSReward: Instituting Standardized Evaluation for Cross-Plat
  [4] 0.3341 2607.28628v1 Learning to Trace Seiberg Dualities
  [5] 0.3300 2607.28575v1 Algorithms for Structured Elections under Thiele Voting Rule

abstained: True
INSUFFICIENT_CONTEXT
The provided sources do not contain information regarding grain yield in Prussia in 1840.


Note what the scores do there. Dense retrieval **always** returns `top_k` — cosine similarity is
a ranking, not a judgment about whether anything is relevant at all. Abstention is therefore the
generator's job in Phase 1, and it stays the generator's job until Phase 6 adds a score
threshold for web escalation. That threshold gets tuned on the golden set, and the scores
printed above are the first data point about where it might sit.

## The CLI

Same objects, no notebook. This is the ROADMAP Phase 1 deliverable — *"end-to-end CLI: question
in, cited answer out."*

```bash
python -m rag.cli "How do multi-agent systems adapt their communication topology?"
python -m rag.cli "what do the figures show about retrieval?" --modality figure --show-context
python -m rag.cli "explain topology adaptation" --category ai -k 8 --json
```

**Release the storage lock first.** Embedded Qdrant allows exactly one process at a time, so the
CLI cannot open the index while this kernel holds it.

In [10]:
retriever.close()
print("Qdrant storage released - the CLI can open it now")

Qdrant storage released - the CLI can open it now


## What Phase 1 still owes

1. **Nothing here is measured.** Every judgment above is a human reading output. The golden set,
   the judge and the four metrics are Phase 4, and ROADMAP §8 is explicit that eval ships
   *before* the agent layer — so this is the last notebook before that.
2. **`ai/` only — 20 of 100 documents.** The other four categories have never been parsed
   (`PHASE1_NOTES.md` §Phase 1 note). Every score above comes from a 544-chunk corpus.
3. **Prompt is version one.** Phase 11 iterates it against the oracle-context arm; if
   faithfulness is low with perfect context, the fault is the prompt and not retrieval.
4. **No conversation.** Single-turn only. Follow-up contextualisation is Phase 5b.
5. **`config/pricing.yaml` is still zeros.** Token counts are recorded per answer, so $/query
   becomes arithmetic the moment real prices land — no re-running anything.